### Fine Tuning Methods for NLP tasks
* Discriminative Fine-Tuning
    * Different layers of a pre-trained model capture different types of information
    * Approach
        * Use differnet learning rates for different layers of the model
        * Lower learning rates for early layers (general features)
        * Higher Learning rates for later layers (task-specific features)
* Slanted Triangular Learning Rates (STLR)
    * Dynamically adjusts learning rates during training to balance exploration and convergence
    * Phases
        * Warm-up: Gradually increase the learning rates to promote exploration
        * Decay: Slowly decrease the learning rate to ensure convergence
* Usecase:
    * Effective for fine tuning pre=trainend models like BERT and GPT

### Regularization and drouput for preventing overfitting in NLPP models
* Regularization
    * L1 Regularization: Encourages sparsity by penalizing absolute weights
    * L2 Regularization (Ridge): Penalizes large weights to improve generalization
* Drouput
    * Randomly Drops units (along with their connections ) during training
    * Prevents over-reliance on specific neurons
    * commonly used in transformer -based models

### Evaluating Model Perfromance with NLP -Specifi Metrics
* Key Metrics:
    * F1 Score
        * 8 Harminiv mean of precision and recall
        * Suitable for classification tasks with imbalanced datasets
    * BLEU score
        * Evaluates the quality of generated text agains refernce text
        * Commonly Used for translation and summarization tasks
    * ROUGE Score
        * Measures overlap between generated and refernce text
        * Used for summarization text
        

In [ ]:
import torch 
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_scheduler
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

In [3]:
dataset = load_dataset("stanfordnlp/imdb")
train_text, test_texts, train_labels, test_labels = train_test_split(
    dataset["train"]["text"], dataset["train"]["label"], test_size=0.2, random_state=42
)

In [6]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
def tokenize_function(texts, labels, tokenizer, max_length=128):
    encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_length)
    return {
        "input_ids":encodings["input_ids"],
        "attention_mask": encodings["attention_mask"],
        "labels": labels
    }

train_data = tokenize_function(train_text, train_labels, tokenizer)
test_data = tokenize_function(test_texts, test_labels, tokenizer)


In [8]:
class IMDBDataset(torch.utils.data.Dataset):
    def __init__(self,data):
        self.input_ids = data["input_ids"]
        self.attention_mask = data["attention_mask"]
        self.labels = data["labels"]

    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self,idx):
        return {
            "input_ids":torch.tensor(self.input_ids[idx]),
            "attention_maxk": torch.tensor(self.attention_mask[idx]),
            "labels": torch.tensor(self.labels[idx])
        }

train_dataaset = IMDBDataset(train_data)
test_dataset = IMDBDataset(test_data)

train_loader = DataLoader(train_dataaset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

In [9]:
# Load Pre Trained Model
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1784.46it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

In [11]:
# Define Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
#  Define Slanted  Triangular Learning Rate
num_training_steps = len(train_loader)*3
warmup_steps = int(0.1* num_training_steps)
scheduler = get_scheduler(
    "slanted_trinangular", optimizer= optimizer, num_warmup_steps=warmup_steps, num_training_steps= num_training_steps,
)



ValueError: slanted_trinangular is not a valid SchedulerType, please select one of ['linear', 'cosine', 'cosine_with_restarts', 'polynomial', 'constant', 'constant_with_warmup', 'inverse_sqrt', 'reduce_lr_on_plateau', 'cosine_with_min_lr', 'cosine_warmup_with_min_lr', 'warmup_stable_decay', 'greedy']

In [ ]:
#  Training Loop
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def train_model():
    model.train()
    for epoch in range(3):
        for batch in train_loader:
            batch = {k:v.to(device) for k,v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
train_model()

            

In [12]:
# Evaluate Model using F1 score
model.eval()
all_preds, all_labels = [],[]

with torch.no_grad():
    for batch in test_loader:
        batch = {k:v.to(device) for k,v in batch.items()}
        outputs = model(**batch)
        preds = torch.argmax(outputs.logits,dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch["labels"].cpu().numpy())

f1 = f1_score(all_labels,all_preds, average="weighted")

print('F1 Score', f1)

NameError: name 'device' is not defined

In [15]:

from sacrebleu import BLEU

refernces = [["this is a test sentence", "this is a smaple sentence"]]
hypothesis = ["this is a test"]

bleu = BLEU()
bleu_score = bleu.corpus_score(hypothesis,refernces)
print("BLEU score", bleu_score)

BLEU score BLEU = 77.88 100.0/100.0/100.0/100.0 (BP = 0.779 ratio = 0.800 hyp_len = 4 ref_len = 5)
